In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append('/content/drive/MyDrive/Colab_Notebooks')  # adjust to your folder

!pip install torch numpy scipy wfdb scikit-learn matplotlib --quiet

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import json
import time
import random
from pathlib import Path
from datetime import datetime
from collections import defaultdict

# ── Numerical ─────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt

# ── PyTorch ───────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# ── Scikit-learn metrics ──────────────────────────────────────────────────────
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix
)

# ── Your modules ──────────────────────────────────────────────────────────────
from PPG_Breath_Model_Preprocessing import (
    Config, SignalProcessor, BIDMCLoader, subject_split
)
from PPG_Breath_Model_Dataset import (
    RespiratoryWindowDataset, make_loader
)
from PPG_Breath_Model_Architecture import RespiratoryDetector

# ── Verify imports worked ─────────────────────────────────────────────────────
print("All imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:             {torch.cuda.get_device_name(0)}")

In [ ]:
cfg = Config()

# ── Signal processing (do not change unless switching datasets) ───────────────
cfg.TARGET_FS    = 100
cfg.WINDOW_S     = 5.0       # seconds per window
cfg.STRIDE_S     = 0.5       # seconds between window starts

# ── Filters (set by physiology, do not change) ───────────────────────────────
cfg.PPG_LOW      = 0.1
cfg.PPG_HIGH     = 0.5
cfg.ECG_LOW      = 1.0
cfg.ECG_HIGH     = 45.0
cfg.FILTER_ORDER = 5

# ── Architecture ──────────────────────────────────────────────────────────────
cfg.ENCODER_CH   = [32, 64]
cfg.CORE_CH      = [64, 128, 128]
cfg.DENSE_UNITS  = 128
cfg.DROPOUT      = 0.3

# ── Loss ──────────────────────────────────────────────────────────────────────
cfg.ALPHA        = 0.90      # weight on classification loss
cfg.POS_WEIGHT   = 2.0       # penalty multiplier for missed transitions

# ── Training dynamics ─────────────────────────────────────────────────────────
cfg.BATCH_SIZE   = 32        # set to 256 when GPU confirmed working
cfg.GRAD_ACCUM   = 1         # set to 8 on GPU for effective batch of 2048
cfg.LR           = 5e-4
cfg.EPOCHS       = 30        # 30 for manual tuning; 60 for final run
cfg.PATIENCE     = 30        # disable early stopping during tuning experiments
cfg.WEIGHT_DECAY = 1e-4

# ── Paths ─────────────────────────────────────────────────────────────────────
cfg.BIDMC_DIR    = "/content/drive/MyDrive/Colab_Notebooks/Data/bidmc-ppg-and-respiration-dataset-1.0.0"
cfg.CKPT_PATH    = "/content/drive/MyDrive/Colab_Notebooks/checkpoints/best.pt"
Path(cfg.CKPT_PATH).parent.mkdir(parents=True, exist_ok=True)

# ── Inference / stitching (tuned after training, not during) ─────────────────
cfg.PROB_THRESHOLD = 0.5
cfg.REFRACTORY_S   = 1.5

# ── Device ────────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Config set:")
print(f"  Window:        {cfg.WINDOW_S}s = {cfg.WINDOW_SAMPLES} samples")
print(f"  Stride:        {cfg.STRIDE_S}s = {cfg.STRIDE_SAMPLES} samples")
print(f"  Batch size:    {cfg.BATCH_SIZE}")
print(f"  Effective batch: {cfg.BATCH_SIZE * cfg.GRAD_ACCUM}")
print(f"  Device:        {device}")

In [ ]:
print("Loading BIDMC recordings...")
bidmc_loader   = BIDMCLoader(cfg)
all_recordings = bidmc_loader.load_all()

if not all_recordings:
    raise RuntimeError(
        f"No recordings loaded. Check cfg.BIDMC_DIR: {cfg.BIDMC_DIR}"
    )

# Subject-level train/val split — 85% train, 15% val
train_recs, val_recs = subject_split(
    all_recordings, val_fraction=0.15, seed=42
)

print(f"\nTotal subjects loaded: {len(all_recordings)}")
print(f"Train subjects:        {len(train_recs)}")
print(f"Val subjects:          {len(val_recs)}")

# Sanity check on signal content
rec0 = train_recs[0]
print(f"\nFirst training recording:")
print(f"  PPG length:    {len(rec0['primary'])} samples "
      f"= {len(rec0['primary'])/cfg.TARGET_FS:.0f}s")
print(f"  Label values:  {np.unique(rec0['labels'])}")
print(f"  Subject ID:    {rec0['subject_id']}")

In [ ]:
def build_loaders(cfg, train_recs, val_recs, num_workers=0):
    """
    Build train and val datasets and DataLoaders from current cfg.
    Returns (train_loader, val_loader, train_ds, val_ds).
    num_workers=0 is safest for Colab; set to 2 on GPU runtime.
    """
    train_ds = RespiratoryWindowDataset(train_recs, cfg, augment=True)
    val_ds   = RespiratoryWindowDataset(val_recs,   cfg, augment=False)

    # Safety check — dataset must be larger than batch size
    # (required because drop_last=True in make_loader)
    assert len(train_ds) > cfg.BATCH_SIZE, (
        f"BATCH_SIZE ({cfg.BATCH_SIZE}) >= train windows ({len(train_ds)}). "
        f"Reduce BATCH_SIZE or increase data."
    )
    assert len(val_ds) > cfg.BATCH_SIZE, (
        f"BATCH_SIZE ({cfg.BATCH_SIZE}) >= val windows ({len(val_ds)}). "
        f"Reduce BATCH_SIZE."
    )

    train_loader = make_loader(train_ds, cfg, train=True,  num_workers=num_workers)
    val_loader   = make_loader(val_ds,   cfg, train=False, num_workers=num_workers)

    # Class balance report
    n_pos = (train_ds._sample_weights > train_ds._sample_weights.mean()).sum()
    print(f"Train windows: {len(train_ds)} "
          f"({n_pos} positive = {100*n_pos/len(train_ds):.1f}%)")
    print(f"Val windows:   {len(val_ds)}")
    print(f"Train batches: {len(train_loader)}  "
          f"Val batches: {len(val_loader)}")

    return train_loader, val_loader, train_ds, val_ds

# Test it with current cfg
train_loader, val_loader, train_ds, val_ds = build_loaders(
    cfg, train_recs, val_recs
)

In [ ]:
class MaskedCompositeLoss(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.alpha          = cfg.ALPHA
        self.window_samples = float(cfg.WINDOW_SAMPLES)   # for normalization
        self.register_buffer(
            "pos_weight",
            torch.full((cfg.NUM_TRANSITIONS,), cfg.POS_WEIGHT)
        )

    def forward(self, preds: dict, targets: dict) -> dict:
        with torch.amp.autocast('cuda', enabled=False):
            t_prob = targets["t_prob"].float()
            t_loc  = targets["t_loc"].float()

            logits_fusion  = preds["logits_fusion"].float()
            logits_primary = preds["logits_primary"].float()
            loc_fusion     = preds["loc"].float()

            bce_fn = nn.BCEWithLogitsLoss(
                pos_weight=self.pos_weight.to(t_prob.device)
            )
            l_cls_fusion  = bce_fn(logits_fusion,  t_prob)
            l_cls_primary = bce_fn(logits_primary, t_prob)
            l_cls = (l_cls_fusion + l_cls_primary) / 2.0

            mask        = (t_loc >= 0.0).float()
            loc_gt_safe = t_loc.clamp(min=0.0)
            n_valid     = mask.sum().clamp(min=1.0)

            # ── Normalize to [0, 1] before squaring ───────────────────────────
            # Max sq error drops from 250,000 to 1.0 — keeps gradient
            # magnitudes well inside fp16 range even after GradScaler's
            # multiplicative scaling
            loc_fusion_norm = loc_fusion / self.window_samples
            loc_gt_norm     = loc_gt_safe / self.window_samples

            #instead of taking square error we will try taking absolute error here

            # sq_err_fusion = (loc_fusion_norm - loc_gt_norm) ** 2
            # l_reg_fusion  = (sq_err_fusion * mask).sum() / n_valid

            abs_err_fusion = torch.abs(loc_fusion_norm - loc_gt_norm)
            l_reg_fusion  = (abs_err_fusion * mask).sum() / n_valid

            # if preds["loc_primary"] is not None:
            #     loc_primary_norm = preds["loc_primary"].float() / self.window_samples
            #     sq_err_primary   = (loc_primary_norm - loc_gt_norm) ** 2
            #     l_reg_primary    = (sq_err_primary * mask).sum() / n_valid
            #     l_reg = (l_reg_fusion + l_reg_primary) / 2.0
            # else:
            #     l_reg = l_reg_fusion

            if preds["loc_primary"] is not None:
                loc_primary_norm = preds["loc_primary"].float() / self.window_samples
                abs_err_primary  = torch.abs(loc_primary_norm - loc_gt_norm)
                l_reg_primary    = (abs_err_primary * mask).sum() / n_valid
                l_reg = (l_reg_fusion + l_reg_primary) / 2.0
            else:
                l_reg = l_reg_fusion

            total = self.alpha * l_cls + (1.0 - self.alpha) * l_reg

        return {"total": total, "cls": l_cls, "reg": l_reg}

In [ ]:
def train_one_epoch(model, loader, optimizer, loss_fn,
                    scaler, cfg, device):
    model.train()
    totals    = defaultdict(float)
    n_batches = len(loader)
    accum     = cfg.GRAD_ACCUM

    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(loader):
        primary     = batch["primary"].to(device,     non_blocking=True)
        auxiliary   = batch["auxiliary"].to(device,   non_blocking=True)
        aux_marker1 = batch["aux_marker1"].to(device, non_blocking=True)
        t_prob      = batch["t_prob"].to(device,      non_blocking=True)
        t_loc       = batch["t_loc"].to(device,       non_blocking=True)

        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            preds       = model(primary, auxiliary, aux_marker1)
            losses      = loss_fn(preds, {"t_prob": t_prob, "t_loc": t_loc})
            scaled_loss = losses["total"] / accum

        scaler.scale(scaled_loss).backward()

        if (step + 1) % accum == 0 or (step + 1) == n_batches:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        for k, v in losses.items():
            totals[k] += v.item()

    return {k: v / n_batches for k, v in totals.items()}


# @torch.no_grad()
# def evaluate(model, loader, loss_fn, device):
#     model.eval()
#     totals    = defaultdict(float)
#     all_probs = []
#     all_labels= []

#     for batch in loader:
#         primary     = batch["primary"].to(device,     non_blocking=True)
#         auxiliary   = batch["auxiliary"].to(device,   non_blocking=True)
#         aux_marker1 = batch["aux_marker1"].to(device, non_blocking=True)
#         t_prob      = batch["t_prob"].to(device,      non_blocking=True)
#         t_loc       = batch["t_loc"].to(device,       non_blocking=True)

#         with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
#             preds  = model(primary, auxiliary, aux_marker1)
#             losses = loss_fn(preds, {"t_prob": t_prob, "t_loc": t_loc})

#         for k, v in losses.items():
#             totals[k] += v.item()

#         all_probs.append(preds["prob"].cpu())
#         all_labels.append(batch["t_prob"])

#     n      = len(loader)
#     losses = {k: v / n for k, v in totals.items()}

#     # Compute F1 across all validation batches
#     probs  = torch.cat(all_probs).numpy()
#     labels = torch.cat(all_labels).numpy()
#     preds_bin = (probs >= 0.5).astype(int)

#     losses["f1_insp"] = f1_score(
#         labels[:, 0], preds_bin[:, 0], zero_division=0
#     )
#     losses["f1_exp"] = f1_score(
#         labels[:, 1], preds_bin[:, 1], zero_division=0
#     )
#     losses["precision_insp"] = precision_score(
#         labels[:, 0], preds_bin[:, 0], zero_division=0
#     )
#     losses["recall_insp"] = recall_score(
#         labels[:, 0], preds_bin[:, 0], zero_division=0
#     )

#     return losses

@torch.no_grad()
def evaluate(model, loader, loss_fn, device, cfg):
    model.eval()
    totals    = defaultdict(float)
    all_probs = []
    all_labels= []

    # ── Tracking variables for MAE ──
    total_mae_ms = 0.0
    valid_regression_batches = 0

    for batch in loader:
        primary     = batch["primary"].to(device,     non_blocking=True)
        auxiliary   = batch["auxiliary"].to(device,   non_blocking=True)
        aux_marker1 = batch["aux_marker1"].to(device, non_blocking=True)
        t_prob      = batch["t_prob"].to(device,      non_blocking=True)
        t_loc       = batch["t_loc"].to(device,       non_blocking=True)

        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            preds  = model(primary, auxiliary, aux_marker1)
            losses = loss_fn(preds, {"t_prob": t_prob, "t_loc": t_loc})

        for k, v in losses.items():
            totals[k] += v.item()

        # ── Calculate Localization MAE in Milliseconds ──
        mask = (t_loc >= 0.0)
        if mask.any():
            abs_err_samples = torch.abs(preds["loc"][mask] - t_loc[mask])
            err_ms = (abs_err_samples / cfg.TARGET_FS) * 1000.0
            total_mae_ms += err_ms.mean().item()
            valid_regression_batches += 1

        all_probs.append(preds["prob"].cpu())
        all_labels.append(batch["t_prob"])

    n      = len(loader)
    losses = {k: v / n for k, v in totals.items()}

    # ── Finalise MAE metric ──
    if valid_regression_batches > 0:
        losses["mae_ms"] = total_mae_ms / valid_regression_batches
    else:
        losses["mae_ms"] = 0.0

    # Compute F1 across all validation batches
    probs  = torch.cat(all_probs).numpy()
    labels = torch.cat(all_labels).numpy()
    preds_bin = (probs >= 0.5).astype(int)

    losses["f1_insp"] = f1_score(labels[:, 0], preds_bin[:, 0], zero_division=0)
    losses["f1_exp"]  = f1_score(labels[:, 1], preds_bin[:, 1], zero_division=0)
    losses["precision_insp"] = precision_score(labels[:, 0], preds_bin[:, 0], zero_division=0)
    losses["recall_insp"] = recall_score(labels[:, 0], preds_bin[:, 0], zero_division=0)

    return losses

In [ ]:
def run_experiment(cfg, train_loader, val_loader,
                   experiment_name="experiment",
                   save_checkpoint=True):
    """
    Run a full training loop and return the epoch-by-epoch history.

    Parameters
    ----------
    cfg              : Config object with all hyperparameters set
    train_loader     : DataLoader for training
    val_loader       : DataLoader for validation
    experiment_name  : string label for logging and plot titles
    save_checkpoint  : whether to save best model weights to cfg.CKPT_PATH

    Returns
    -------
    history : dict of lists, one entry per epoch
        Keys: train_cls, train_reg, train_total,
              val_cls, val_reg, val_total,
              val_f1_insp, val_f1_exp,
              val_precision_insp, val_recall_insp, lr
    """
    print(f"\n{'═'*65}")
    print(f"  Experiment: {experiment_name}")
    print(f"  ALPHA={cfg.ALPHA}  POS_WEIGHT={cfg.POS_WEIGHT}  "
          f"DROPOUT={cfg.DROPOUT}  LR={cfg.LR:.2e}")
    print(f"  ENCODER_CH={cfg.ENCODER_CH}  CORE_CH={cfg.CORE_CH}  "
          f"DENSE_UNITS={cfg.DENSE_UNITS}")
    print(f"  BATCH={cfg.BATCH_SIZE}  ACCUM={cfg.GRAD_ACCUM}  "
          f"EPOCHS={cfg.EPOCHS}")
    print(f"{'═'*65}\n")

    # Fresh model, loss, optimizer, scheduler for every experiment
    model   = RespiratoryDetector(cfg).to(device)
    dummy_input = torch.zeros(2, 1, cfg.WINDOW_SAMPLES, device=device)
    _ = model(dummy_input, dummy_input, dummy_input)
    loss_fn = MaskedCompositeLoss(cfg).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr           = cfg.LR,
        betas        = (0.9, 0.99),
        weight_decay = cfg.WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=cfg.EPOCHS, eta_min=cfg.LR / 10
    )
    scaler = torch.amp.GradScaler(
        'cuda', enabled=(device.type == 'cuda')
    )

    history  = defaultdict(list)
    best_val = float("inf")
    patience_count = 0

    for epoch in range(1, cfg.EPOCHS + 1):
        t0 = time.time()

        tr = train_one_epoch(
            model, train_loader, optimizer, loss_fn, scaler, cfg, device
        )
        va = evaluate(model, val_loader, loss_fn, device, cfg)

        scheduler.step()
        lr_now  = scheduler.get_last_lr()[0]
        elapsed = time.time() - t0

        # Record history
        history["train_cls"].append(tr["cls"])
        history["train_reg"].append(tr["reg"])
        history["train_total"].append(tr["total"])
        history["val_mae_ms"].append(va["mae_ms"])
        history["val_cls"].append(va["cls"])
        history["val_reg"].append(va["reg"])
        history["val_total"].append(va["total"])
        history["val_f1_insp"].append(va["f1_insp"])
        history["val_f1_exp"].append(va["f1_exp"])
        history["val_precision_insp"].append(va["precision_insp"])
        history["val_recall_insp"].append(va["recall_insp"])
        history["lr"].append(lr_now)

        print(
            f"Ep {epoch:3d}/{cfg.EPOCHS} | "
            f"tr_cls={tr['cls']:.4f} tr_reg={tr['reg']:.4f} | "
            f"va_cls={va['cls']:.4f} va_reg={va['reg']:.4f} MAE={va['mae_ms']:.4f}ms | "
            f"F1 i={va['f1_insp']:.4f} e={va['f1_exp']:.4f} | "
            f"P={va['precision_insp']:.4f} R={va['recall_insp']:.4f} | "
            f"lr={lr_now:.2e} {elapsed:.4f}s"
        )

        # Checkpoint on best validation loss
        if save_checkpoint and va["total"] < best_val:
            best_val = va["total"]
            patience_count = 0
            ckpt_path = Path(cfg.CKPT_PATH).with_stem(
                Path(cfg.CKPT_PATH).stem + f"_{experiment_name}"
            )
            torch.save({
                "epoch":       epoch,
                "model_state": model.state_dict(),
                "optim_state": optimizer.state_dict(),
                "val_loss":    best_val,
                "cfg":         cfg.__dict__,
                "experiment":  experiment_name,
            }, ckpt_path)
            print(f"  ✓ Checkpoint saved (val={best_val:.4f})")
        else:
            patience_count += 1
            if cfg.PATIENCE < cfg.EPOCHS and patience_count >= cfg.PATIENCE:
                print(f"\n  Early stop at epoch {epoch}")
                break

    # Log to file
    log_experiment(experiment_name, cfg, history)

    return dict(history), model

In [ ]:
LOG_PATH = "/content/drive/MyDrive/Colab_Notebooks/experiment_log.jsonl"

def log_experiment(name, cfg, history):
    """Append experiment summary to the permanent log file."""
    best_epoch = int(np.argmax(history["val_f1_insp"])) + 1

    record = {
        "timestamp":      datetime.now().isoformat(),
        "experiment":     name,
        "hyperparameters": {
            "ALPHA":        cfg.ALPHA,
            "POS_WEIGHT":   cfg.POS_WEIGHT,
            "DROPOUT":      cfg.DROPOUT,
            "LR":           cfg.LR,
            "WEIGHT_DECAY": cfg.WEIGHT_DECAY,
            "BATCH_SIZE":   cfg.BATCH_SIZE,
            "GRAD_ACCUM":   cfg.GRAD_ACCUM,
            "ENCODER_CH":   cfg.ENCODER_CH,
            "CORE_CH":      cfg.CORE_CH,
            "DENSE_UNITS":  cfg.DENSE_UNITS,
            "WINDOW_S":     cfg.WINDOW_S,
            "EPOCHS_RUN":   len(history["val_total"]),
        },
        "results": {
            "best_val_f1_insp": float(max(history["val_f1_insp"])),
            "best_val_f1_exp":  float(max(history["val_f1_exp"])),
            "best_epoch":       best_epoch,
            "final_val_cls":    float(history["val_cls"][-1]),
            "final_val_reg":    float(history["val_reg"][-1]),
            "final_precision":  float(history["val_precision_insp"][-1]),
            "final_recall":     float(history["val_recall_insp"][-1]),
        }
    }

    with open(LOG_PATH, "a") as f:
        f.write(json.dumps(record) + "\n")

    print(f"\n  Logged → best F1_insp={record['results']['best_val_f1_insp']:.3f} "
          f"at epoch {best_epoch}")

In [ ]:
def plot_history(history, experiment_name=""):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    epochs = range(1, len(history["val_total"]) + 1)

    # Plot 1: Classification loss (overfitting diagnostic)
    axes[0].plot(epochs, history["train_cls"], label="train cls", color="steelblue")
    axes[0].plot(epochs, history["val_cls"],   label="val cls",   color="steelblue",
                 linestyle="--")
    axes[0].axhline(y=0.693, color="red", linestyle=":", alpha=0.5,
                    label="random baseline (0.693)")
    axes[0].set_title("Classification Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend(fontsize=8)
    axes[0].grid(True, alpha=0.3)

    # Plot 2: F1 scores
    axes[1].plot(epochs, history["val_f1_insp"], label="Inspiration F1",
                 color="green")
    axes[1].plot(epochs, history["val_f1_exp"],  label="Expiration F1",
                 color="orange")
    axes[1].set_title("Validation F1 Score")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylim(0, 1)
    axes[1].legend(fontsize=8)
    axes[1].grid(True, alpha=0.3)

    # Plot 3: Precision vs Recall
    axes[2].plot(epochs, history["val_precision_insp"],
                 label="Precision (insp)", color="purple")
    axes[2].plot(epochs, history["val_recall_insp"],
                 label="Recall (insp)",    color="crimson")
    axes[2].set_title("Precision vs Recall (Inspiration)")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylim(0, 1)
    axes[2].legend(fontsize=8)
    axes[2].grid(True, alpha=0.3)

    fig.suptitle(experiment_name, fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(
        f"/content/drive/MyDrive/Colab_Notebooks/plots/{experiment_name}.png",
        dpi=120, bbox_inches="tight"
    )
    plt.show()

# Create plots folder
Path("/content/drive/MyDrive/Colab_Notebooks/plots").mkdir(
    parents=True, exist_ok=True
)

In [ ]:
model = RespiratoryDetector(cfg).to(device)

# Check BEFORE any forward pass — dense layers don't exist yet
print(f"fusion_core._dense before forward: {model.fusion_core._dense}")

batch = next(iter(train_loader))
p = batch["primary"].to(device)
e = batch["auxiliary"].to(device)
r = batch["aux_marker1"].to(device)

with torch.amp.autocast('cuda', enabled=True):
    feat_ppg   = model.primary_encoder(p)
    feat_ecg   = model.support_encoder(e)
    feat_rpeak = model.feature_encoder(r)
    fused      = torch.cat([feat_ppg, feat_ecg, feat_rpeak], dim=1)

    # Run JUST the conv portion of fusion_core, before dense is built
    conv_out = model.fusion_core.convs(fused)
    print(f"\nflat_dim (input to first Linear): {conv_out.shape[1]}")
    print(f"conv_out range: [{conv_out.min().item():.1f}, {conv_out.max().item():.1f}]")

    # NOW trigger the lazy dense build
    core_out = model.fusion_core(fused)
    print(f"\nfusion_core._dense exists now: {model.fusion_core._dense is not None}")

    # Inspect the raw output of the FIRST linear layer, before BatchNorm
    first_linear = model.fusion_core._dense[0]
    print(f"First Linear weight init std: {first_linear.weight.std().item():.6f}")

    raw_linear_out = first_linear(conv_out)
    print(f"Raw first-Linear output range: "
          f"[{raw_linear_out.min().item():.1f}, {raw_linear_out.max().item():.1f}]  "
          f"dtype={raw_linear_out.dtype}")
    print(f"Contains inf: {torch.isinf(raw_linear_out).any().item()}")
    print(f"Contains nan: {torch.isnan(raw_linear_out).any().item()}")

In [ ]:
import torch
import torch.nn as nn

model = RespiratoryDetector(cfg).to(device)
model.train()   # match real training conditions, not eval()

batch = next(iter(train_loader))
p = batch["primary"].to(device)
e = batch["auxiliary"].to(device)
r = batch["aux_marker1"].to(device)
t_prob = batch["t_prob"].to(device)
t_loc  = batch["t_loc"].to(device)

def check(name, t):
    if t is None:
        print(f"  {name:20s}: None")
        return
    n_nan = torch.isnan(t).sum().item()
    n_inf = torch.isinf(t).sum().item()
    flag  = "  ← BAD" if (n_nan or n_inf) else ""
    print(f"  {name:20s}: dtype={str(t.dtype):15s} "
          f"range=[{t.float().min().item():.3f}, {t.float().max().item():.3f}]  "
          f"NaN={n_nan} Inf={n_inf}{flag}")

print("── Full forward pass, every output ──")
with torch.amp.autocast('cuda', enabled=True):
    out = model(p, e, r)

for k, v in out.items():
    check(k, v)

print("\n── Loss, term by term ──")
loss_fn = MaskedCompositeLoss(cfg).to(device)

with torch.amp.autocast('cuda', enabled=True):
    bce_fn = nn.BCEWithLogitsLoss(pos_weight=loss_fn.pos_weight.to(device))
    l_cls_fusion  = bce_fn(out["logits_fusion"],  t_prob)
    l_cls_primary = bce_fn(out["logits_primary"], t_prob)
    check("l_cls_fusion", l_cls_fusion)
    check("l_cls_primary", l_cls_primary)

    mask        = (t_loc >= 0.0).float()
    loc_gt_safe = t_loc.clamp(min=0.0)
    n_valid     = mask.sum().clamp(min=1.0)

    sq_err_fusion = (out["loc"] - loc_gt_safe) ** 2
    check("sq_err_fusion", sq_err_fusion)
    l_reg_fusion = (sq_err_fusion * mask).sum() / n_valid
    check("l_reg_fusion", l_reg_fusion)

    sq_err_primary = (out["loc_primary"] - loc_gt_safe) ** 2
    check("sq_err_primary", sq_err_primary)
    l_reg_primary = (sq_err_primary * mask).sum() / n_valid
    check("l_reg_primary", l_reg_primary)

    total = cfg.ALPHA * ((l_cls_fusion + l_cls_primary)/2) + \
            (1-cfg.ALPHA) * ((l_reg_fusion + l_reg_primary)/2)
    check("total", total)

print("\n── Backward pass + gradient check ──")
scaler = torch.amp.GradScaler('cuda', enabled=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR)
optimizer.zero_grad(set_to_none=True)

scaler.scale(total).backward()
scaler.unscale_(optimizer)

bad_grads = []
for name, param in model.named_parameters():
    if param.grad is not None:
        if torch.isnan(param.grad).any() or torch.isinf(param.grad).any():
            bad_grads.append(name)

if bad_grads:
    print(f"  Bad gradients found in: {bad_grads}")
else:
    print("  All gradients finite")

print(f"\n  GradScaler scale factor: {scaler.get_scale()}")
scaler.step(optimizer)
scaler.update()
print(f"  Scale factor after step: {scaler.get_scale()}")

# Check if the optimizer step itself corrupted any parameter
bad_params = []
for name, param in model.named_parameters():
    if torch.isnan(param).any() or torch.isinf(param).any():
        bad_params.append(name)

if bad_params:
    print(f"  Parameters corrupted after step: {bad_params}")
else:
    print("  All parameters finite after optimizer step")

In [ ]:
loss_fn = MaskedCompositeLoss(cfg).to(device)   # uses the fixed version above

with torch.amp.autocast('cuda', enabled=True):
    out    = model(p, e, r)
    losses = loss_fn(out, {"t_prob": t_prob, "t_loc": t_loc})

print(f"total: {losses['total'].item():.4f}")
print(f"cls:   {losses['cls'].item():.4f}")
print(f"reg:   {losses['reg'].item():.4f}")   # should now be small, e.g. 0.05–0.3

scaler = torch.amp.GradScaler('cuda', enabled=True)
scaler.scale(losses["total"]).backward()
scaler.unscale_(optimizer)

bad = [n for n, p in model.named_parameters()
       if p.grad is not None and (torch.isnan(p.grad).any() or torch.isinf(p.grad).any())]
print(f"Bad gradients: {bad if bad else 'none — all clean'}")
print(f"Scale factor: {scaler.get_scale()}")

In [ ]:
import inspect

print("── Is the normalization actually in the class? ──")
source = inspect.getsource(MaskedCompositeLoss.forward)
print("Contains 'window_samples':", "window_samples" in source)
print("Contains '/ self.window_samples':", "/ self.window_samples" in source)

print("\n── Does loss_fn have the attribute? ──")
print("loss_fn has window_samples attr:", hasattr(loss_fn, "window_samples"))
if hasattr(loss_fn, "window_samples"):
    print("Value:", loss_fn.window_samples)

In [ ]:
cfg.ALPHA      = 0.90
cfg.POS_WEIGHT = 2.0
cfg.DROPOUT    = 0.3
cfg.LR         = 5e-4
cfg.EPOCHS     = 30
cfg.PATIENCE   = 30   # disable early stopping during tuning

train_loader, val_loader, _, _ = build_loaders(cfg, train_recs, val_recs)

history_baseline, _ = run_experiment(
    cfg, train_loader, val_loader,
    experiment_name="E0_baseline"
)

plot_history(history_baseline, "E0 — Baseline")

In [ ]:
import numpy as np

# Assuming val_recs and cfg are already loaded
S = cfg.STRIDE_SAMPLES

for win_s in [3.0,2.9,2.8,2.7,2.6,2.5,2.4,2.3,2.2,2.1,2.0]:
    W = int(win_s * cfg.TARGET_FS)

    n_with_insp = 0
    n_with_exp  = 0
    n_total     = 0

    for rec in val_recs:
        labels = rec["labels"]
        T      = len(labels)
        for start in range(0, T - W + 1, S):
            window = labels[start:start+W]
            n_total += 1
            if (window == 1).any():
                n_with_insp += 1
            if (window == 2).any():
                n_with_exp += 1

    print(f"\nWindow Size: {win_s}s ({W} samples)")
    print(f"  Total val windows:        {n_total}")
    print(f"  Windows with inspiration: {n_with_insp} ({100*n_with_insp/n_total:.1f}%)")
    print(f"  Windows with expiration:  {n_with_exp} ({100*n_with_exp/n_total:.1f}%)")

In [ ]:
# Define the range of window sizes to test (in seconds)
window_sizes = [2.0, 3.0, 4.0, 5.0]

# Target overlap fraction to maintain ensemble voting power
# 0.90 ensures Window / Stride = 10 overlapping views per transition
overlap_fraction = 0.90

# Keep tracking dictionaries for downstream analysis
sweep_histories = {}

# Keep your established hyperparameter baselines
cfg.ALPHA      = 0.90
cfg.POS_WEIGHT = 2.0
cfg.DROPOUT    = 0.3
cfg.LR         = 5e-4
cfg.EPOCHS     = 30
cfg.PATIENCE   = 30   # disable early stopping during tuning

for win_s in window_sizes:
    # Dynamically calculate stride to maintain constant voting power
    stride_s = win_s * (1.0 - overlap_fraction)

    # 1. Update the configuration profile
    cfg.WINDOW_S = win_s
    cfg.STRIDE_S = stride_s

    exp_name = f"Sweep_Win{win_s:.1f}s_Stride{stride_s:.2f}s"

    print(f"\n{'='*70}")
    print(f" INITIALIZING {exp_name}")
    print(f" Overlap Views: {win_s / stride_s:.1f}")
    print(f"{'='*70}")

    # 2. Re-generate datasets and loaders (mandatory because sequence length changes)
    train_loader, val_loader, train_ds, val_ds = build_loaders(cfg, train_recs, val_recs)

    # 3. Execute training
    # (run_experiment automatically instantiates a fresh model and optimizer)
    history, best_model = run_experiment(
        cfg,
        train_loader,
        val_loader,
        experiment_name=exp_name,
        save_checkpoint=True  # Saves to cfg.CKPT_PATH with exp_name appended
    )

    # 4. Evaluate and store
    plot_history(history, exp_name)
    sweep_histories[exp_name] = history

In [ ]:
def test_saved_checkpoint(ckpt_filename, cfg, val_recs, device):
    ckpt_path = Path(cfg.CKPT_PATH).parent / ckpt_filename
    if not ckpt_path.exists():
        print(f"Checkpoint not found: {ckpt_path}")
        return

    print(f"\nEvaluating: {ckpt_filename}")

    # Load the checkpoint to retrieve the exact config used during training
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)

    # Restore the specific Window and Stride used for this checkpoint
    cfg.WINDOW_S = ckpt["cfg"]["WINDOW_S"]
    cfg.STRIDE_S = ckpt["cfg"]["STRIDE_S"]

    # Rebuild the dataloader so the tensor sizes match the window
    _, val_loader, _, _ = build_loaders(cfg, train_recs, val_recs, num_workers=0)

    # Rebuild model
    model = RespiratoryDetector(cfg).to(device)

    # ── CRITICAL FIX: Trigger lazy initialization ──
    # Create a dummy tensor shaped (Batch=1, Channels=1, WindowSamples)
    dummy_input = torch.zeros(2, 1, cfg.WINDOW_SAMPLES, device=device)

    # Run a silent forward pass to build the _dense layers
    with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
        _ = model(dummy_input, dummy_input, dummy_input)
    # ───────────────────────────────────────────────

    # Now it is safe to load the weights
    model.load_state_dict(ckpt["model_state"])

    loss_fn = MaskedCompositeLoss(cfg).to(device)

    # Run the updated evaluation
    va = evaluate(model, val_loader, loss_fn, device, cfg)

    print(f"  F1 Inspiration: {va['f1_insp']:.3f}")
    print(f"  F1 Expiration:  {va['f1_exp']:.3f}")
    print(f"  Regression MAE: {va['mae_ms']:.1f} ms")


test_saved_checkpoint("best_Sweep_Win2.0s_Stride0.20s.pt", cfg, val_recs, device)

In [ ]:
test_saved_checkpoint("best_Sweep_Win3.0s_Stride0.30s.pt", cfg, val_recs, device)

In [ ]:
test_saved_checkpoint("best_E0_baseline.pt", cfg, val_recs, device)

In [ ]:
# Define the range of window sizes to test (in seconds)
window_sizes = [3.0]

# Target overlap fraction to maintain ensemble voting power
# 0.90 ensures Window / Stride = 10 overlapping views per transition
overlap_fraction = 0.90

# Keep tracking dictionaries for downstream analysis
sweep_histories = {}

# Keep your established hyperparameter baselines
cfg.ALPHA      = 0.50
cfg.POS_WEIGHT = 2.0
cfg.DROPOUT    = 0.3
cfg.LR         = 5e-4
cfg.EPOCHS     = 30
cfg.PATIENCE   = 30   # disable early stopping during tuning

for win_s in window_sizes:
    # Dynamically calculate stride to maintain constant voting power
    stride_s = win_s * (1.0 - overlap_fraction)

    # 1. Update the configuration profile
    cfg.WINDOW_S = win_s
    cfg.STRIDE_S = stride_s

    exp_name = f"Sweep_Win{win_s:.1f}s_Stride{stride_s:.2f}s"

    print(f"\n{'='*70}")
    print(f" INITIALIZING {exp_name}")
    print(f" Overlap Views: {win_s / stride_s:.1f}")
    print(f"{'='*70}")

    # 2. Re-generate datasets and loaders (mandatory because sequence length changes)
    train_loader, val_loader, train_ds, val_ds = build_loaders(cfg, train_recs, val_recs)

    # 3. Execute training
    # (run_experiment automatically instantiates a fresh model and optimizer)
    history, best_model = run_experiment(
        cfg,
        train_loader,
        val_loader,
        experiment_name=exp_name,
        save_checkpoint=True  # Saves to cfg.CKPT_PATH with exp_name appended
    )

    # 4. Evaluate and store
    plot_history(history, exp_name)
    sweep_histories[exp_name] = history